In [1]:
import json
import sys
import re
from pathlib import Path
from difflib import SequenceMatcher

import pandas as pd
import numpy as np

BASE_DIR = Path.cwd().parent

if str(BASE_DIR) not in sys.path:
    sys.path.insert(0, str(BASE_DIR))

DATA_PATH = BASE_DIR / "data" / "corpus.json"
RESULTS_PATH = BASE_DIR / "reports" / "extracted_action_items.csv"

with open(DATA_PATH, "r", encoding="utf-8") as f:
    corpus = json.load(f)

print("Corpus loaded")
print("Meetings:", len(corpus["reunions"]))

Corpus loaded
Meetings: 8


In [2]:
if RESULTS_PATH.exists():

    predictions_df = pd.read_csv(
        RESULTS_PATH
    )

    print(
        "Predictions loaded:",
        len(predictions_df)
    )

else:

    predictions_df = pd.DataFrame()

    print(
        "Prediction file not found."
    )

Predictions loaded: 54


In [3]:
ground_truth_records = []

for meeting_index, meeting in enumerate(
    corpus["reunions"]
):

    meeting_id = meeting_index + 1

    terrain = meeting.get(
        "verite_terrain",
        {}
    )

    actions = terrain.get(
        "actions",
        []
    )

    for action in actions:

        ground_truth_records.append({
            "meeting_id": meeting_id,
            "code": action.get("code"),
            "anchor": action.get("ancre"),
            "family": action.get("famille"),
            "raw_action": action
        })

ground_truth_df = pd.DataFrame(
    ground_truth_records
)

print(
    "Ground-truth actions:",
    len(ground_truth_df)
)

display(
    ground_truth_df.head(10)
)

Ground-truth actions: 40


,meeting_id,code,anchor,family,raw_action
0,1,A1,kiosk,attribuee,"{'code': 'A1', 'ancre': 'kiosk', 'famille': 'a..."
1,1,A2,awning,attribuee,"{'code': 'A2', 'ancre': 'awning', 'famille': '..."
2,1,A3,banner,attribuee,"{'code': 'A3', 'ancre': 'banner', 'famille': '..."
3,1,N1,keypad,non-attribuee,"{'code': 'N1', 'ancre': 'keypad', 'famille': '..."
4,1,N2,mailbox,non-attribuee,"{'code': 'N2', 'ancre': 'mailbox', 'famille': ..."
5,2,A1,hydrant,attribuee,"{'code': 'A1', 'ancre': 'hydrant', 'famille': ..."
6,2,A2,mezzanine,attribuee,"{'code': 'A2', 'ancre': 'mezzanine', 'famille'..."
7,2,A3,payslips,attribuee,"{'code': 'A3', 'ancre': 'payslips', 'famille':..."
8,2,N1,labels,non-attribuee,"{'code': 'N1', 'ancre': 'labels', 'famille': '..."
9,2,N2,hose,non-attribuee,"{'code': 'N2', 'ancre': 'hose', 'famille': 'no..."


In [4]:
if ground_truth_records:

    print(
        "Example ground-truth action:"
    )

    print(
        ground_truth_records[0]["raw_action"]
    )

Example ground-truth action:
{'code': 'A1', 'ancre': 'kiosk', 'famille': 'attribuee', 'quoi': 'update the pricing on the kiosk', 'responsable': 'Rafael', 'echeance': 'Wednesday'}


In [5]:
trap_records = []

for meeting_index, meeting in enumerate(
    corpus["reunions"]
):

    meeting_id = meeting_index + 1

    terrain = meeting.get(
        "verite_terrain",
        {}
    )

    for key in [
        "actions_attribuees",
        "actions_non_attribuees"
    ]:

        trap_actions = terrain.get(
            key,
            []
        )

        if isinstance(
            trap_actions,
            list
        ):

            for action in trap_actions:

                if isinstance(
                    action,
                    dict
                ):

                    trap_records.append({
                        "meeting_id": meeting_id,
                        "type": key,
                        "code": action.get("code"),
                        "anchor": action.get("ancre"),
                        "family": action.get("famille"),
                        "raw_action": action
                    })

trap_df = pd.DataFrame(
    trap_records
)

print(
    "Trap records found:",
    len(trap_df)
)

Trap records found: 0


In [6]:
def normalize_text(text):

    if text is None:
        return ""

    text = str(text).lower()

    text = re.sub(
        r"[^a-z0-9\s]",
        " ",
        text
    )

    text = re.sub(
        r"\s+",
        " ",
        text
    )

    return text.strip()

In [7]:
def similarity(a, b):

    a = normalize_text(a)
    b = normalize_text(b)

    if not a or not b:
        return 0.0

    if a in b or b in a:
        return 1.0

    return SequenceMatcher(
        None,
        a,
        b
    ).ratio()


def prediction_matches_anchor(
    prediction,
    anchor
):

    if not anchor:
        return False

    task = prediction.get(
        "task",
        ""
    )

    evidence = prediction.get(
        "evidence",
        ""
    )

    anchor = normalize_text(
        anchor
    )

    task_text = normalize_text(
        task
    )

    evidence_text = normalize_text(
        evidence
    )

    if anchor in task_text:
        return True

    if anchor in evidence_text:
        return True

    if similarity(
        anchor,
        task_text
    ) >= 0.70:

        return True

    if similarity(
        anchor,
        evidence_text
    ) >= 0.70:

        return True

    return False

In [8]:
evaluation_records = []

for meeting_id in range(
    1,
    len(corpus["reunions"]) + 1
):

    gt_meeting = ground_truth_df[
        ground_truth_df["meeting_id"] == meeting_id
    ]

    pred_meeting = predictions_df[
        predictions_df["meeting_id"] == meeting_id
    ] if not predictions_df.empty else pd.DataFrame()

    used_predictions = set()

    for gt_index, gt in gt_meeting.iterrows():

        matched = False
        matched_prediction = None

        for pred_index, prediction in pred_meeting.iterrows():

            if pred_index in used_predictions:
                continue

            if prediction_matches_anchor(
                prediction,
                gt["anchor"]
            ):

                matched = True
                matched_prediction = pred_index

                used_predictions.add(
                    pred_index
                )

                break

        evaluation_records.append({
            "meeting_id": meeting_id,
            "anchor": gt["anchor"],
            "family": gt["family"],
            "matched": matched,
            "prediction_index": matched_prediction
        })

evaluation_df = pd.DataFrame(
    evaluation_records
)

display(
    evaluation_df.head(20)
)

,meeting_id,anchor,family,matched,prediction_index
0,1,kiosk,attribuee,True,1.0
1,1,awning,attribuee,True,3.0
2,1,banner,attribuee,True,2.0
3,1,keypad,non-attribuee,True,7.0
4,1,mailbox,non-attribuee,True,4.0
5,2,hydrant,attribuee,False,NaN
6,2,mezzanine,attribuee,True,14.0
7,2,payslips,attribuee,True,11.0
8,2,labels,non-attribuee,True,13.0
9,2,hose,non-attribuee,True,8.0


In [9]:
true_positives = int(
    evaluation_df["matched"].sum()
)

false_negatives = (
    len(evaluation_df)
    - true_positives
)

total_predictions = (
    len(predictions_df)
    if not predictions_df.empty
    else 0
)

false_positives = max(
    0,
    total_predictions - true_positives
)

precision = (
    true_positives /
    (true_positives + false_positives)
    if (
        true_positives +
        false_positives
    ) > 0
    else 0
)

recall = (
    true_positives /
    (true_positives + false_negatives)
    if (
        true_positives +
        false_negatives
    ) > 0
    else 0
)

f1 = (
    2 * precision * recall /
    (precision + recall)
    if precision + recall > 0
    else 0
)

print(
    "True Positives:",
    true_positives
)

print(
    "False Positives:",
    false_positives
)

print(
    "False Negatives:",
    false_negatives
)

print(
    f"Precision: {precision:.2%}"
)

print(
    f"Recall: {recall:.2%}"
)

print(
    f"F1 Score: {f1:.2%}"
)

True Positives: 30
False Positives: 24
False Negatives: 10
Precision: 55.56%
Recall: 75.00%
F1 Score: 63.83%


In [10]:
meeting_results = []

for meeting_id in range(
    1,
    len(corpus["reunions"]) + 1
):

    meeting_eval = evaluation_df[
        evaluation_df["meeting_id"] == meeting_id
    ]

    tp = int(
        meeting_eval["matched"].sum()
    )

    gt_count = len(
        meeting_eval
    )

    pred_count = (
        len(
            predictions_df[
                predictions_df["meeting_id"] == meeting_id
            ]
        )
        if not predictions_df.empty
        else 0
    )

    fn = gt_count - tp
    fp = max(
        0,
        pred_count - tp
    )

    meeting_precision = (
        tp / (tp + fp)
        if tp + fp > 0
        else 0
    )

    meeting_recall = (
        tp / (tp + fn)
        if tp + fn > 0
        else 0
    )

    meeting_f1 = (
        2 * meeting_precision * meeting_recall /
        (meeting_precision + meeting_recall)
        if meeting_precision + meeting_recall > 0
        else 0
    )

    meeting_results.append({
        "meeting_id": meeting_id,
        "ground_truth": gt_count,
        "predictions": pred_count,
        "true_positives": tp,
        "false_positives": fp,
        "false_negatives": fn,
        "precision": meeting_precision,
        "recall": meeting_recall,
        "f1": meeting_f1
    })

meeting_results_df = pd.DataFrame(
    meeting_results
)

display(
    meeting_results_df
)

,meeting_id,ground_truth,predictions,true_positives,false_positives,false_negatives,precision,recall,f1
0,1,5,8,5,3,0,0.625000,1.0,0.769231
1,2,5,7,4,3,1,0.571429,0.8,0.666667
2,3,5,5,3,2,2,0.600000,0.6,0.600000
3,4,5,6,3,3,2,0.500000,0.6,0.545455
4,5,5,6,3,3,2,0.500000,0.6,0.545455
5,6,5,7,4,3,1,0.571429,0.8,0.666667
6,7,5,8,5,3,0,0.625000,1.0,0.769231
7,8,5,7,3,4,2,0.428571,0.6,0.500000


In [11]:
metrics_df = pd.DataFrame({
    "Metric": [
        "Precision",
        "Recall",
        "F1 Score"
    ],
    "Score": [
        precision,
        recall,
        f1
    ]
})

metrics_df["Score"] = metrics_df[
    "Score"
].map(
    lambda x: f"{x:.2%}"
)

display(
    metrics_df
)

,Metric,Score
0,Precision,55.56%
1,Recall,75.00%
2,F1 Score,63.83%


In [12]:
owner_results = []

if not predictions_df.empty:

    matched_rows = evaluation_df[
        evaluation_df["matched"]
    ]

    for _, row in matched_rows.iterrows():

        pred_index = row[
            "prediction_index"
        ]

        gt_index = ground_truth_df[
            (
                ground_truth_df["meeting_id"]
                == row["meeting_id"]
            )
            &
            (
                ground_truth_df["anchor"]
                == row["anchor"]
            )
        ].index[0]

        prediction = predictions_df.loc[
            pred_index
        ]

        ground_truth = ground_truth_df.loc[
            gt_index
        ]

        owner_results.append({
            "meeting_id": row["meeting_id"],
            "anchor": row["anchor"],
            "predicted_owner": prediction.get(
                "owner"
            ),
            "ground_truth_owner": None
        })

owner_df = pd.DataFrame(
    owner_results
)

print(
    "Matched owner records:",
    len(owner_df)
)

display(
    owner_df.head(20)
)

Matched owner records: 30


,meeting_id,anchor,predicted_owner,ground_truth_owner
0,1,kiosk,NaN,None
1,1,awning,Nadia,None
2,1,banner,NaN,None
3,1,keypad,NaN,None
4,1,mailbox,NaN,None
5,2,mezzanine,NaN,None
6,2,payslips,Iris,None
7,2,labels,Also,None
8,2,hose,NaN,None
9,3,shredder,NaN,None


In [13]:
if not predictions_df.empty:

    owner_count = predictions_df[
        "owner"
    ].notna().sum()

    total_count = len(
        predictions_df
    )

    owner_coverage = (
        owner_count /
        total_count
        if total_count > 0
        else 0
    )

else:

    owner_coverage = 0

print(
    f"Owner coverage: {owner_coverage:.2%}"
)

Owner coverage: 37.04%


In [14]:
if not predictions_df.empty:

    deadline_count = predictions_df[
        "deadline"
    ].notna().sum()

    total_count = len(
        predictions_df
    )

    deadline_coverage = (
        deadline_count /
        total_count
        if total_count > 0
        else 0
    )

else:

    deadline_coverage = 0

print(
    f"Deadline coverage: {deadline_coverage:.2%}"
)

Deadline coverage: 14.81%


In [15]:
if not predictions_df.empty:

    confidence_mean = predictions_df[
        "confidence"
    ].mean()

    confidence_median = predictions_df[
        "confidence"
    ].median()

    print(
        f"Average confidence: {confidence_mean:.2%}"
    )

    print(
        f"Median confidence: {confidence_median:.2%}"
    )

else:

    print(
        "No confidence values available."
    )

Average confidence: 82.41%
Median confidence: 75.00%


In [16]:
if not predictions_df.empty:

    low_confidence_df = predictions_df[
        predictions_df["confidence"] < 0.70
    ]

    print(
        "Low-confidence predictions:",
        len(low_confidence_df)
    )

    display(
        low_confidence_df
    )

else:

    low_confidence_df = pd.DataFrame()

    print(
        "No predictions available."
    )

Low-confidence predictions: 0


,meeting_id,task,owner,deadline,status,confidence,evidence,warnings


In [17]:
duplicate_count = 0

if not predictions_df.empty:

    tasks = predictions_df[
        "task"
    ].fillna("").astype(str).str.lower()

    duplicate_count = (
        tasks.duplicated().sum()
    )

print(
    "Duplicate predictions:",
    duplicate_count
)

Duplicate predictions: 13


In [18]:
summary = {
    "ground_truth_actions": len(
        ground_truth_df
    ),
    "predicted_actions": len(
        predictions_df
    ),
    "true_positives": true_positives,
    "false_positives": false_positives,
    "false_negatives": false_negatives,
    "precision": precision,
    "recall": recall,
    "f1_score": f1,
    "owner_coverage": owner_coverage,
    "deadline_coverage": deadline_coverage,
    "duplicate_predictions": duplicate_count
}

summary_df = pd.DataFrame(
    [summary]
)

display(
    summary_df
)

,ground_truth_actions,predicted_actions,true_positives,false_positives,false_negatives,precision,recall,f1_score,owner_coverage,deadline_coverage,duplicate_predictions
0,40,54,30,24,10,0.555556,0.75,0.638298,0.37037,0.148148,13


In [19]:
REPORTS_DIR = BASE_DIR / "reports"

REPORTS_DIR.mkdir(
    exist_ok=True
)

evaluation_path = (
    REPORTS_DIR /
    "evaluation_results.csv"
)

meeting_results_df.to_csv(
    evaluation_path,
    index=False
)

metrics_path = (
    REPORTS_DIR /
    "evaluation_metrics.csv"
)

metrics_df.to_csv(
    metrics_path,
    index=False
)

summary_path = (
    REPORTS_DIR /
    "evaluation_summary.csv"
)

summary_df.to_csv(
    summary_path,
    index=False
)

print(
    "Evaluation reports saved."
)

Evaluation reports saved.


In [20]:
print("=" * 60)
print("AI MEETING ACTION-ITEM EXTRACTOR")
print("EVALUATION REPORT")
print("=" * 60)

print(
    f"Ground-truth actions : {len(ground_truth_df)}"
)

print(
    f"Predicted actions    : {len(predictions_df)}"
)

print(
    f"True positives       : {true_positives}"
)

print(
    f"False positives      : {false_positives}"
)

print(
    f"False negatives      : {false_negatives}"
)

print(
    f"Precision            : {precision:.2%}"
)

print(
    f"Recall               : {recall:.2%}"
)

print(
    f"F1 Score             : {f1:.2%}"
)

print(
    f"Owner coverage       : {owner_coverage:.2%}"
)

print(
    f"Deadline coverage    : {deadline_coverage:.2%}"
)

print(
    f"Duplicate predictions: {duplicate_count}"
)

print("=" * 60)

AI MEETING ACTION-ITEM EXTRACTOR
EVALUATION REPORT
Ground-truth actions : 40
Predicted actions    : 54
True positives       : 30
False positives      : 24
False negatives      : 10
Precision            : 55.56%
Recall               : 75.00%
F1 Score             : 63.83%
Owner coverage       : 37.04%
Deadline coverage    : 14.81%
Duplicate predictions: 13
